# Flow Matching

Diffusion models take 20~50 sampling steps because they walk a curved path from noise to data. Flow matching and rectified flow trained straight paths. Straighter paths means fewer steps mean faster inference.

## Problem Definition

DDPM's reverse process is a 1000-step stochastic walk from `N(0, I)` back to the data distribution. DDIM collapsed it to 20-50 deterministic steps. You want fewer steps -- ideally one. The block is that the ODE solving the reverse process if stiff, the path is curved.

If you could train the model such that the path from noise to data was a straight line, a single Euler step from `t=1` to `t=0` would work. Flow matching builds this directly: define a straight-line interpolation from `x_1 ~ N(0, I)` to `x_0 ~ data`, train a vector field `v_0(x, t)` to match its time derivative, integrate at inference.

Rectified flow goes further: Iteratively straighten the paths with a reflow procedure that produces a progressively closer-to-linear ODE. After two reflow iterations, a 2-step sampler matches 50-step DDPM quality.

## Basic Concept

Flow matching: Straight-line interpolation between noise and data

### Straight-line flow

```
x_t = t * x_1 + (1 - t) * x_0   t ~ [0, 1]
```

where `x_0 ~ data` and `x_1 ~ N(0, I)`. The time derivative along this straight line is constant:

```
dx_t / dt = x_1 - x_0
```

Define a neural vector field `v_0(v_t, t)` and train it to match this derivative:

```
L = E_{x_0, x_1, t} ||v_0(x_t, t) - (x_1 - x_0)||^2
```

This is the **conditional flow matching** loss.

### Sampling

At inference, integrate the learned vector field backwards in time:
```
x_{t-delta_t} = x_t - delta_t * v_0(x_t, t)
```

Start at `x_t ~ N(0, I)`, Euler-step down to `t = 0`

### Rectified flow

Straight-line flow works but the learned paths are not actually straight -- they curve because many `x_0`s can map to the same `x_1`. Rectified flows's reflow step.

1. Train flow model `v_1` with random pairings.
2. Sample N pairs `(x_1, x_0)` by integrating `v_1` from `x_1` to its landing `x_0`
3. Tran `v_2` on those paired examples. Because the pairs are now "ODE-matched", the straight-line interpolant between them is genuinely flatter.
4. Repeat.

### Why it won

1. Simulation-free training.   no ODE unrolling during training, trivial to implement.
2. Better loss geometry.  Straight paths have consistent signal-to-noise, where as DDPM epsilon-loss has bad SNR at edges of the schedule.
3. Faster inference.  4-8 steps at SDXL-Turbo quality. 1 step with consistency distllation.

### Flow matching VS DDPM

Flow mathcing with a Gaussian-conditional path is diffusion with a specific noise schudual. Pick the `x_t = alpht(t) x_0 + sigma(t) x_1` schedule and flow matching recovers Stratonovich-reformulated diffusion with `v = alpha' * x_0 - sigma' * x_1`. The two are algebrically equivalent for Gaussian paths.

# Build your Own

In [8]:
import torch
import torch.nn as nn
import math

torch.manual_seed(42)

class FlowMatchingNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x, t):
        # x, t: (B,) -> feats: (B, 5)
        t = t.reshape(-1)
        x = x.reshape(-1)
        feats = torch.stack([
            x,
            t,
            t * t,
            torch.sin(2 * math.pi * t),
            torch.cos(2 * math.pi * t),
        ], dim=-1)
        return self.net(feats).squeeze(-1)

net = FlowMatchingNet(5, 24, 1)
opt = torch.optim.Adam(net.parameters(), lr=1e-3)

def sample_data(n):
    mean = torch.where(torch.rand(n) < 0.5, -2.0, 2.0)
    return torch.normal(mean, 0.3)

batch_size = 256
for step in range(2000):
    x_0 = sample_data(batch_size)
    x_1 = torch.randn(batch_size)
    t = torch.rand(batch_size)  # must be U(0,1), not randn

    x_t = t * x_1 + (1 - t) * x_0
    pred = net(x_t, t)
    target = x_1 - x_0
    loss = ((pred - target) ** 2).mean()

    opt.zero_grad()
    loss.backward()
    opt.step()

    if (step + 1) % 200 == 0:
        print(f"Step {step + 1}: Loss={loss.item():.4f}")



Step 200: Loss=3.2258
Step 400: Loss=3.2562
Step 600: Loss=3.2078
Step 800: Loss=2.5709
Step 1000: Loss=2.9953
Step 1200: Loss=2.4591
Step 1400: Loss=3.0086
Step 1600: Loss=2.4664
Step 1800: Loss=2.6567
Step 2000: Loss=2.3182


## Sample

Euler-integrate the learned vector field from noise (`t=1`) down to data (`t=0`), then compare the generated histogram with the true bimodal mixture.

In [ ]:
import matplotlib.pyplot as plt

@torch.no_grad()
def sample(net, n=2000, steps=50):
    """Integrate dx/dt = v(x,t) backward from t=1 (noise) to t=0 (data)."""
    x = torch.randn(n)
    dt = 1.0 / steps
    traj = [x.clone()]
    for i in range(steps):
        t = torch.full((n,), 1.0 - i * dt)
        x = x - dt * net(x, t)
        traj.append(x.clone())
    return x, torch.stack(traj)  # (n,), (steps+1, n)

n_gen = 2000
gen, traj = sample(net, n=n_gen, steps=50)
real = sample_data(n_gen)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

# left: histogram real vs generated
bins = torch.linspace(-4, 4, 60).numpy()
axes[0].hist(real.numpy(), bins=bins, density=True, alpha=0.55, label="real data")
axes[0].hist(gen.numpy(), bins=bins, density=True, alpha=0.55, label="generated")
axes[0].set_title("Distribution: real vs generated")
axes[0].set_xlabel("x")
axes[0].legend()

# right: a few ODE trajectories from noise -> data
show = 40
t_grid = torch.linspace(1.0, 0.0, traj.shape[0]).numpy()
for i in range(show):
    axes[1].plot(t_grid, traj[:, i].numpy(), color="C0", alpha=0.25, linewidth=1)
axes[1].set_title(f"ODE trajectories (n={show})")
axes[1].set_xlabel("t")
axes[1].set_ylabel("x")
axes[1].invert_xaxis()  # t: 1 -> 0 left to right feels natural; keep math axis clear

plt.tight_layout()
plt.show()

print(f"generated mean={gen.mean():.3f}, std={gen.std():.3f}")
print(f"real      mean={real.mean():.3f}, std={real.std():.3f}")
